<a href="https://colab.research.google.com/github/nawresssjebali/building_directory/blob/main/BCP_SCRAPING_CLEANING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Discover all postcode outcodes (districts) covering Greater London and Essex,
using postcodes.io's nearest-outcodes endpoint tiled over a grid.

Why: rather than guessing town/borough names, this systematically enumerates
real postcode districts within the two regions' approximate bounding boxes,
using the same authoritative source (postcodes.io) already used elsewhere
in the BodyOwn geocoding pipeline.

Method:
1. Define approximate bounding boxes for Greater London and Essex.
2. Lay a grid of points (~8km spacing) across each box.
3. Query postcodes.io /outcodes?lon=&lat=&limit=100 at each grid point.
4. Union all returned outcodes, de-duplicated.
5. Save the combined outcode list to a JSON/CSV file for use as scraper seeds.

Caveat: the bounding boxes below are approximate rectangles covering the
regions (not official boundary polygons), so this will include some outcodes
just outside Greater London/Essex proper (e.g. bordering Kent, Hertfordshire,
Suffolk) and may need manual trimming after review.
"""

import requests
import csv
import time

POSTCODES_IO_OUTCODES_ENDPOINT = "https://api.postcodes.io/outcodes"

# Approximate bounding boxes (lat_min, lat_max, lon_min, lon_max)
# Greater London: roughly the M25 extent
# Essex: county extent (excluding London boroughs already covered above)
BOUNDING_BOXES = {
    "greater_london": (51.28, 51.70, -0.51, 0.33),
    "essex": (51.50, 52.10, 0.00, 1.30),
}

GRID_SPACING_KM = 8  # spacing between grid points; smaller = more thorough, more API calls
KM_PER_DEGREE_LAT = 111.0  # approx


def build_grid(lat_min, lat_max, lon_min, lon_max, spacing_km):
    """Generate a grid of (lat, lon) points across a bounding box."""
    lat_step = spacing_km / KM_PER_DEGREE_LAT
    # longitude degrees shrink with latitude; approximate using mid-latitude
    mid_lat = (lat_min + lat_max) / 2
    import math
    lon_step = spacing_km / (KM_PER_DEGREE_LAT * math.cos(math.radians(mid_lat)))

    points = []
    lat = lat_min
    while lat <= lat_max:
        lon = lon_min
        while lon <= lon_max:
            points.append((round(lat, 4), round(lon, 4)))
            lon += lon_step
        lat += lat_step
    return points


def fetch_outcodes_near(lat, lon, limit=100, radius=25000):
    """Query postcodes.io for outcodes near a lat/lon point.
    radius is in meters; 25000 (25km) is the API's documented max."""
    params = {"lon": lon, "lat": lat, "limit": limit, "radius": radius}
    resp = requests.get(POSTCODES_IO_OUTCODES_ENDPOINT, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    if data.get("status") != 200:
        return []
    result = data.get("result") or []
    return [item["outcode"] for item in result]


def discover_all_outcodes(region_name, bbox, spacing_km=GRID_SPACING_KM, delay=0.3):
    lat_min, lat_max, lon_min, lon_max = bbox
    grid_points = build_grid(lat_min, lat_max, lon_min, lon_max, spacing_km)
    print(f"[{region_name}] grid has {len(grid_points)} points")

    all_outcodes = set()
    for i, (lat, lon) in enumerate(grid_points, 1):
        try:
            outcodes = fetch_outcodes_near(lat, lon)
            all_outcodes.update(outcodes)
        except requests.RequestException as e:
            print(f"  ERROR at ({lat},{lon}): {e}")
        if i % 10 == 0:
            print(f"  [{i}/{len(grid_points)}] running total: {len(all_outcodes)} outcodes")
        time.sleep(delay)

    return sorted(all_outcodes)


def save_outcodes_csv(outcode_map, path="london_essex_outcodes.csv"):
    """outcode_map: dict of region_name -> list of outcodes"""
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["region", "outcode"])
        for region, outcodes in outcode_map.items():
            for oc in outcodes:
                writer.writerow([region, oc])
    print(f"Saved outcode list to {path}")


if __name__ == "__main__":
    outcode_map = {}
    for region, bbox in BOUNDING_BOXES.items():
        outcode_map[region] = discover_all_outcodes(region, bbox)
        print(f"[{region}] TOTAL unique outcodes: {len(outcode_map[region])}")

    save_outcodes_csv(outcode_map)

    # Print combined de-duplicated list for direct use as scraper seeds
    combined = sorted(set().union(*outcode_map.values()))
    print(f"\nCombined unique outcodes across both regions: {len(combined)}")
    print(combined)

[greater_london] grid has 48 points
  [10/48] running total: 232 outcodes
  [20/48] running total: 313 outcodes
  [30/48] running total: 404 outcodes
  [40/48] running total: 463 outcodes
[greater_london] TOTAL unique outcodes: 490
[essex] grid has 108 points
  [10/108] running total: 205 outcodes
  [20/108] running total: 235 outcodes
  [30/108] running total: 259 outcodes
  [40/108] running total: 285 outcodes
  [50/108] running total: 300 outcodes
  [60/108] running total: 305 outcodes
  [70/108] running total: 320 outcodes
  [80/108] running total: 325 outcodes
  [90/108] running total: 336 outcodes
  [100/108] running total: 342 outcodes
[essex] TOTAL unique outcodes: 347
Saved outcode list to london_essex_outcodes.csv

Combined unique outcodes across both regions: 595
['AL1', 'AL10', 'AL2', 'AL3', 'AL4', 'AL5', 'AL6', 'AL7', 'AL8', 'AL9', 'BR1', 'BR2', 'BR3', 'BR4', 'BR5', 'BR6', 'BR7', 'BR8', 'CB1', 'CB10', 'CB11', 'CB2', 'CB21', 'CB22', 'CB23', 'CB24', 'CB25', 'CB3', 'CB4', 'CB

In [ ]:
"""
Filter the raw outcode list (from discover_outcodes.py) down to postcode
AREAS that genuinely belong to Greater London or Essex, per Royal Mail's
official postcode area definitions.

Why this is needed: the grid-tiling approach in discover_outcodes.py used a
25km search radius per point, which spilled well past the two regions'
true extent and picked up neighboring counties (Herts, Kent, Surrey, Berks,
Bucks, Suffolk, Sussex, Cambridgeshire).

Reference: Royal Mail's "London postal area" comprises the postcode areas
E, EC, N, NW, SE, SW, W, WC (inner) plus the outer-London-serving areas
BR, CR, DA, EN, HA, IG, KT, RM, SM, TW, UB, WD. Essex-specific areas are
CM, CO, SS (IG and RM straddle the London/Essex border and are already
included above).

Caveat (still worth knowing): several of the "outer" areas above extend
administratively beyond Greater London's actual boundary -- e.g. WD is
mostly Hertfordshire (Watford), EN and DA both dip into Herts/Kent, KT and
TW touch Surrey. They're kept here because they're part of the *postal*
area historically served from London, which is what matters for a
location-based teacher search -- but if you want strict administrative
-boundary accuracy later, this is the point to revisit.
"""

import csv

# Royal Mail postcode AREA prefixes (the letters before the first digit)
# genuinely associated with Greater London or Essex.
LONDON_AREAS = {"E", "EC", "N", "NW", "SE", "SW", "W", "WC",
                 "BR", "CR", "DA", "EN", "HA", "IG", "KT", "RM",
                 "SM", "TW", "UB", "WD"}
ESSEX_AREAS = {"CM", "CO", "SS"}  # IG, RM already in LONDON_AREAS (border areas)

ALLOWED_AREAS = LONDON_AREAS | ESSEX_AREAS


def outcode_area(outcode):
    """Extract the letter prefix (postcode area) from an outcode, e.g. 'EC1A' -> 'EC', 'CM1' -> 'CM'."""
    area = ""
    for ch in outcode:
        if ch.isalpha():
            area += ch
        else:
            break
    return area


def filter_outcodes(input_csv="london_essex_outcodes.csv", output_csv="london_essex_outcodes_filtered.csv"):
    kept = []
    dropped = []

    with open(input_csv, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            outcode = row["outcode"]
            area = outcode_area(outcode)
            if area in ALLOWED_AREAS:
                kept.append(outcode)
            else:
                dropped.append(outcode)

    kept = sorted(set(kept))
    dropped = sorted(set(dropped))

    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["outcode"])
        for oc in kept:
            writer.writerow([oc])

    print(f"Kept {len(kept)} outcodes (Greater London + Essex postal areas)")
    print(f"Dropped {len(dropped)} outcodes from neighboring counties")
    print(f"\nDropped areas found: {sorted(set(outcode_area(o) for o in dropped))}")
    print(f"\nSaved filtered list to {output_csv}")

    return kept, dropped


if __name__ == "__main__":
    kept, dropped = filter_outcodes()
    print("\n--- KEPT (use these as scraper seeds) ---")
    print(kept)

Kept 410 outcodes (Greater London + Essex postal areas)
Dropped 185 outcodes from neighboring counties

Dropped areas found: ['AL', 'CB', 'CT', 'GU', 'HP', 'IP', 'ME', 'RG', 'RH', 'SG', 'SL', 'TN']

Saved filtered list to london_essex_outcodes_filtered.csv

--- KEPT (use these as scraper seeds) ---
['BR1', 'BR2', 'BR3', 'BR4', 'BR5', 'BR6', 'BR7', 'BR8', 'CM0', 'CM1', 'CM11', 'CM12', 'CM13', 'CM14', 'CM15', 'CM16', 'CM17', 'CM18', 'CM19', 'CM2', 'CM20', 'CM21', 'CM22', 'CM23', 'CM24', 'CM3', 'CM4', 'CM5', 'CM6', 'CM7', 'CM77', 'CM8', 'CM9', 'CM92', 'CM98', 'CM99', 'CO1', 'CO10', 'CO11', 'CO12', 'CO13', 'CO14', 'CO15', 'CO16', 'CO2', 'CO3', 'CO4', 'CO5', 'CO6', 'CO7', 'CO8', 'CO9', 'CR0', 'CR2', 'CR3', 'CR4', 'CR5', 'CR6', 'CR7', 'CR8', 'CR9', 'CR90', 'DA1', 'DA10', 'DA11', 'DA12', 'DA13', 'DA14', 'DA15', 'DA16', 'DA17', 'DA18', 'DA2', 'DA3', 'DA4', 'DA5', 'DA6', 'DA7', 'DA8', 'DA9', 'E1', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E1W', 'E2', 'E20', 'E22', 'E3', 'E

In [ ]:
"""
Body Control Pilates Register scraper -- fast/concurrent version
Source: https://www.bodycontrolpilates.com/wp-content/plugins/bcp-teacher-finder/ajax/bcp-tf-results.php
Filter: Q003 = "Pre- & Postnatal Pilates" (Layer 2 specialism)

Run this in your own environment (not this sandbox) -- the sandbox network
allowlist doesn't include bodycontrolpilates.com or postcodes.io.

WHAT CHANGED VS THE ORIGINAL, AND WHY IT'S FASTER
--------------------------------------------------
1. Concurrency (ThreadPoolExecutor): this workload is 100% I/O wait (network
   round trips), so running MAX_WORKERS requests in flight at once gives close
   to an N-times speedup for free -- the CPU is idle the whole time anyway.

2. A shared rate limiter (token-bucket-ish, via a lock + timestamp) replaces
   `time.sleep(1.5)` after every request. Before: strictly serial, 1.5s dead
   time per request no matter what. Now: up to MAX_REQUESTS_PER_SEC requests
   can be in flight, and the limiter just makes sure you don't exceed that
   *rate* -- you're still being polite to their server, but you're not idling
   the whole pipeline to do it.

3. requests.Session() with an HTTPAdapter pool: reuses TCP/TLS connections
   instead of paying a fresh handshake for every single request.

4. Incremental + resumable output: results are appended to the CSV as they
   come in (not held in memory and written once at the end), and completed
   seeds are tracked in a small `.done` file. If the script dies partway
   through a few hundred outcodes, re-running it skips everything already
   done instead of starting over.

5. Cap-refinement (the sub-search-by-postcode step) is submitted as more
   jobs into the *same* thread pool instead of a nested serial loop, so it
   doesn't create a second serial bottleneck on top of the first.

Tune MAX_WORKERS and MAX_REQUESTS_PER_SEC below. Start conservative (this
config is deliberately modest) and only raise it if you're not seeing
errors/429s from BCP's server.
"""

import csv
import os
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter

ENDPOINT = "https://www.bodycontrolpilates.com/wp-content/plugins/bcp-teacher-finder/ajax/bcp-tf-results.php"
HEADERS = {
    "Content-Type": "application/x-www-form-urlencoded",
    "User-Agent": "Mozilla/5.0 (compatible; BodyOwnDirectoryResearch/1.0)",
}
FILTER_PRE_POSTNATAL = "Q003,"
CAP_THRESHOLD = 50

OUTPUT_CSV = "bcp_pilates_prepostnatal.csv"
DONE_FILE = "bcp_pilates_prepostnatal.done"  # tracks which seeds are fully processed

MAX_WORKERS = 8              # concurrent in-flight requests
MAX_REQUESTS_PER_SEC = 4.0   # aggregate rate cap across all workers, be polite

FIELDNAMES = [
    "tf_id", "name", "address", "phone_raw", "email", "website",
    "qualifications", "has_pre_postnatal", "distance", "search_seed",
]


# ---------------------------------------------------------------------------
# Rate limiter shared across worker threads
# ---------------------------------------------------------------------------
class RateLimiter:
    def __init__(self, max_per_sec):
        self.min_interval = 1.0 / max_per_sec
        self._lock = threading.Lock()
        self._next_slot = time.monotonic()

    def wait(self):
        with self._lock:
            now = time.monotonic()
            if self._next_slot < now:
                self._next_slot = now
            wait_time = self._next_slot - now
            self._next_slot += self.min_interval
        if wait_time > 0:
            time.sleep(wait_time)


rate_limiter = RateLimiter(MAX_REQUESTS_PER_SEC)

_session = requests.Session()
_adapter = HTTPAdapter(pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS)
_session.mount("https://", _adapter)
_session.mount("http://", _adapter)


# ---------------------------------------------------------------------------
# Resumable CSV writer -- one lock, append-as-you-go
# ---------------------------------------------------------------------------
_write_lock = threading.Lock()
_seen_ids_lock = threading.Lock()
_seen_ids = set()


def load_done_seeds():
    if not os.path.exists(DONE_FILE):
        return set()
    with open(DONE_FILE, encoding="utf-8") as f:
        return set(line.strip() for line in f if line.strip())


def mark_seed_done(seed):
    with _write_lock:
        with open(DONE_FILE, "a", encoding="utf-8") as f:
            f.write(seed + "\n")


def ensure_csv_header():
    if not os.path.exists(OUTPUT_CSV):
        with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=FIELDNAMES).writeheader()


def append_records(records):
    if not records:
        return
    with _write_lock:
        with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
            for r in records:
                writer.writerow(r)


def load_seen_ids_from_existing_csv():
    """So a resumed run doesn't re-append duplicates if a seed partially wrote."""
    if not os.path.exists(OUTPUT_CSV):
        return
    with open(OUTPUT_CSV, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row.get("tf_id"):
                _seen_ids.add(row["tf_id"])


# ---------------------------------------------------------------------------
# Network + parsing (same parsing logic as original)
# ---------------------------------------------------------------------------
def fetch_results(location, filters=FILTER_PRE_POSTNATAL, country="GB"):
    if not location.upper().endswith("UK"):
        location = f"{location}, UK"
    payload = {
        "searchtype": "nearest",
        "country": country,
        "location": location,
        "filters": filters,
    }
    rate_limiter.wait()
    resp = _session.post(ENDPOINT, data=payload, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    return resp.text


def parse_teacher_cards(html):
    soup = BeautifulSoup(html, "html.parser")
    results = []

    for card in soup.select("div.tf-single"):
        record = {}
        record["tf_id"] = card.get("id", "").replace("tf-", "")

        distance_el = card.select_one("h3.tf-distance")
        record["distance"] = distance_el.get_text(strip=True) if distance_el else None

        cols = card.select("div.col")
        if len(cols) < 2:
            continue

        col1 = cols[0]
        name_el = col1.select_one("h3")
        record["name"] = name_el.get_text(strip=True) if name_el else None

        p_el = col1.select_one("p")
        if p_el:
            address_lines = [line.strip() for line in p_el.get_text(separator="|").split("|") if line.strip()]
            record["address"] = ", ".join(address_lines)
        else:
            record["address"] = None

        website_el = col1.select_one('a[href^="http"]')
        record["website"] = website_el["href"] if website_el else None

        email_el = col1.select_one('a[href^="mailto:"]')
        record["email"] = email_el["href"].replace("mailto:", "") if email_el else None

        phone_candidates = [t.strip() for t in col1.find_all(string=True, recursive=False)]
        phone_candidates = [t for t in phone_candidates if any(c.isdigit() for c in t)]
        record["phone_raw"] = phone_candidates[0] if phone_candidates else None

        col2 = cols[1]
        qual_el = col2.select_one("div.col-qual")
        if qual_el:
            quals = [q.strip() for q in qual_el.get_text(separator="|").split("|") if q.strip()]
            record["qualifications"] = "; ".join(quals)
            record["has_pre_postnatal"] = any("Pre- & Postnatal Pilates" in q for q in quals)
        else:
            record["qualifications"] = ""
            record["has_pre_postnatal"] = False

        results.append(record)

    return results


def get_sample_postcodes_for_outcode(outcode, limit=20):
    rate_limiter.wait()
    resp = requests.get(
        "https://api.postcodes.io/postcodes",
        params={"q": outcode, "limit": limit},
        timeout=10,
    )
    resp.raise_for_status()
    data = resp.json()
    if data.get("status") != 200:
        return []
    return [item["postcode"] for item in (data.get("result") or [])]


def load_seed_locations(path="london_essex_outcodes_filtered.csv"):
    seeds = []
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            seeds.append(row["outcode"])
    return seeds


# ---------------------------------------------------------------------------
# Worker jobs
# ---------------------------------------------------------------------------
def process_seed(loc):
    """Fetch one seed, dedupe + write new records, return (loc, new_count, hit_cap)."""
    try:
        html = fetch_results(loc)
        records = parse_teacher_cards(html)
    except requests.RequestException as e:
        print(f"  ERROR fetching {loc}: {e}")
        return loc, 0, False

    new_records = []
    with _seen_ids_lock:
        for r in records:
            if r["tf_id"] not in _seen_ids:
                _seen_ids.add(r["tf_id"])
                r["search_seed"] = loc
                new_records.append(r)

    append_records(new_records)
    hit_cap = len(records) >= CAP_THRESHOLD
    return loc, len(new_records), hit_cap


def refine_capped_seed(loc):
    """Sub-search individual postcodes within an outcode that hit the cap."""
    total_new = 0
    try:
        postcodes = get_sample_postcodes_for_outcode(loc)
    except requests.RequestException as e:
        print(f"  ERROR getting postcodes for {loc}: {e}")
        return loc, 0

    for pc in postcodes:
        _, new_count, _ = process_seed(pc)
        total_new += new_count
    return loc, total_new


def scrape_all(seed_locations):
    ensure_csv_header()
    load_seen_ids_from_existing_csv()
    done = load_done_seeds()
    remaining = [s for s in seed_locations if s not in done]
    print(f"{len(done)} seeds already done, {len(remaining)} remaining")

    capped_seeds = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(process_seed, loc): loc for loc in remaining}
        for i, future in enumerate(as_completed(futures), 1):
            loc = futures[future]
            try:
                _, new_count, hit_cap = future.result()
            except Exception as e:
                print(f"  UNEXPECTED ERROR for {loc}: {e}")
                continue
            print(f"[{i}/{len(remaining)}] {loc}: +{new_count} new" + (" (CAP HIT)" if hit_cap else ""))
            mark_seed_done(loc)
            if hit_cap:
                capped_seeds.append(loc)

    if capped_seeds:
        print(f"\nRefining {len(capped_seeds)} capped seeds by postcode...")
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futures = {pool.submit(refine_capped_seed, loc): loc for loc in capped_seeds}
            for future in as_completed(futures):
                loc, total_new = future.result()
                print(f"  refined {loc}: +{total_new} new from sub-postcodes")

    print(f"\nDone. Results in {OUTPUT_CSV}")


if __name__ == "__main__":
    seeds = load_seed_locations()
    print(f"Loaded {len(seeds)} seed locations from filtered outcode list")
    scrape_all(seeds)

Loaded 410 seed locations from filtered outcode list
0 seeds already done, 410 remaining
[1/410] BR2: +50 new (CAP HIT)
[2/410] BR3: +7 new (CAP HIT)
[3/410] BR1: +5 new (CAP HIT)
[4/410] BR6: +4 new (CAP HIT)
[5/410] BR5: +0 new (CAP HIT)
[6/410] BR4: +0 new (CAP HIT)
[7/410] BR7: +1 new (CAP HIT)
[8/410] BR8: +2 new (CAP HIT)
[9/410] CM0: +43 new (CAP HIT)
[10/410] CM1: +16 new (CAP HIT)
[11/410] CM11: +0 new (CAP HIT)
[12/410] CM12: +0 new (CAP HIT)
[13/410] CM13: +0 new (CAP HIT)
[14/410] CM14: +5 new (CAP HIT)
[15/410] CM15: +0 new (CAP HIT)
[16/410] CM16: +8 new (CAP HIT)
[17/410] CM17: +4 new (CAP HIT)
[18/410] CM18: +0 new (CAP HIT)
[19/410] CM19: +4 new (CAP HIT)
[20/410] CM2: +0 new (CAP HIT)
[21/410] CM20: +1 new (CAP HIT)
[22/410] CM21: +10 new (CAP HIT)
[23/410] CM22: +3 new (CAP HIT)
[24/410] CM23: +1 new (CAP HIT)
[25/410] CM24: +1 new (CAP HIT)
[26/410] CM3: +0 new (CAP HIT)
[27/410] CM4: +0 new (CAP HIT)
[28/410] CM5: +0 new (CAP HIT)
[29/410] CM6: +0 new (CAP HIT)
[30

In [ ]:
import csv

with open("bcp_pilates_prepostnatal.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    rows = [r for r in reader if r["has_pre_postnatal"] == "True"]

with open("bcp_pilates_prepostnatal_filtered.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

In [ ]:
import csv

with open("bcp_pilates_prepostnatal_filtered.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    fieldnames = [f for f in reader.fieldnames if f != "tf_id"]
    rows = list(reader)

seen = set()
deduped = []
for r in rows:
    r.pop("tf_id", None)
    key = tuple(r.values())
    if key not in seen:
        seen.add(key)
        deduped.append(r)

with open("bcp_pilates_prepostnatal_final.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(deduped)

In [ ]:
# ── Dedupe Pilates instructor CSV into a directory-ready xlsx ──
# Run in Google Colab. Upload your CSV when prompted.

# Cell 1 — install/import
!pip install openpyxl pandas -q

import pandas as pd
import re
from google.colab import files
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter

# Cell 2 — upload the CSV
uploaded = files.upload()  # pick your bcp_pilates_prepostnatal CSV
csv_name = list(uploaded.keys())[0]
df = pd.read_csv(csv_name)

# Cell 3 — normalize + build a dedupe key
def norm_name(n):
    return re.sub(r'\s+', ' ', str(n)).strip()

def norm_email(e):
    return None if pd.isna(e) else str(e).strip().lower()

def norm_phone(p):
    return None if pd.isna(p) else re.sub(r'\D', '', str(p))

df['name_norm'] = df['name'].apply(norm_name)
df['email_norm'] = df['email'].apply(norm_email)
df['phone_norm'] = df['phone_raw'].apply(norm_phone)

def make_key(row):
    e, p = row['email_norm'], row['phone_norm']
    if pd.notna(e) and e:
        return 'email:' + str(e)
    if pd.notna(p) and p:
        return 'phone:' + str(p)
    return 'name:' + row['name_norm'].lower()

df['key'] = df.apply(make_key, axis=1)
print(f"Unique instructors: {df['key'].nunique()} (from {len(df)} rows)")

# Cell 4 — collapse duplicates, split Layer 1 / Layer 2 qualifications
rows = []
for key, g in df.groupby('key'):
    name = g['name_norm'].value_counts().index[0]
    phone = next((p for p in g['phone_raw'] if pd.notna(p)), '')
    email = next((e for e in g['email'] if pd.notna(e)), '')
    website = next((w for w in g['website'] if pd.notna(w) and str(w).strip()), '')

    all_quals = set()
    for q in g['qualifications']:
        for part in q.split(';'):
            all_quals.add(part.strip())
    layer2 = 'Pre- & Postnatal Pilates' in all_quals
    layer1_quals = sorted(all_quals - {'Pre- & Postnatal Pilates'})

    addresses = list(dict.fromkeys(g['address'].tolist()))
    seeds = sorted(set(g['search_seed'].tolist()))
    min_dist = g['distance'].apply(lambda x: float(str(x).replace(' KM', '').strip())).min()

    rows.append({
        'Instructor Name': name,
        'Phone': phone,
        'Email': email,
        'Website': website,
        'Layer 1 - Base Qualifications': '; '.join(layer1_quals),
        'Layer 2 - Pre/Postnatal Specialism': 'Verified' if layer2 else 'Not Verified',
        'Verification Source': 'Body Control Pilates directory',
        'Number of Locations': len(addresses),
        'Locations': ' | '.join(addresses),
        'Nearest Distance (KM)': min_dist,
        'Search Postcodes Covered': ', '.join(seeds),
    })

out = pd.DataFrame(rows).sort_values('Instructor Name').reset_index(drop=True)

# Cell 5 — write a formatted xlsx
wb = Workbook()
ws = wb.active
ws.title = "Pilates Instructors"

headers = list(out.columns)
ws.append(headers)

header_font = Font(name='Arial', bold=True, color='FFFFFF', size=11)
header_fill = PatternFill(start_color='2E5E4E', end_color='2E5E4E', fill_type='solid')
for col_idx in range(1, len(headers) + 1):
    cell = ws.cell(row=1, column=col_idx)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(vertical='center', wrap_text=True)

for _, row in out.iterrows():
    ws.append(list(row))

body_font = Font(name='Arial', size=10)
for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    for cell in row:
        cell.font = body_font
        cell.alignment = Alignment(vertical='top', wrap_text=True)

widths = {
    'Instructor Name': 20, 'Phone': 16, 'Email': 28, 'Website': 30,
    'Layer 1 - Base Qualifications': 32, 'Layer 2 - Pre/Postnatal Specialism': 20,
    'Verification Source': 24, 'Number of Locations': 12, 'Locations': 60,
    'Nearest Distance (KM)': 12, 'Search Postcodes Covered': 22
}
for col_idx, h in enumerate(headers, 1):
    ws.column_dimensions[get_column_letter(col_idx)].width = widths.get(h, 18)

ws.freeze_panes = "A2"
ws.auto_filter.ref = f"A1:{get_column_letter(len(headers))}{ws.max_row}"

out_name = "Pilates_Directory_Deduped.xlsx"
wb.save(out_name)

# Cell 6 — download
files.download(out_name)

Saving bcp_pilates_prepostnatal_final.csv to bcp_pilates_prepostnatal_final (1).csv
Unique instructors: 150 (from 313 rows)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ── Check which instructor websites are actually live ──
# Run in Google Colab. Upload your deduped xlsx when prompted.

# Cell 1 — install/import
!pip install requests openpyxl pandas -q

import pandas as pd
import requests
from google.colab import files
from concurrent.futures import ThreadPoolExecutor, as_completed

# Cell 2 — upload the deduped xlsx
uploaded = files.upload()  # pick Pilates_Directory_Deduped.xlsx
xlsx_name = list(uploaded.keys())[0]
df = pd.read_excel(xlsx_name)

# Cell 3 — check each website
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}

def check_url(url, timeout=10):
    if pd.isna(url) or not str(url).strip():
        return "No URL"
    u = str(url).strip()
    if not u.startswith("http"):
        u = "https://" + u
    try:
        r = requests.get(u, headers=HEADERS, timeout=timeout, allow_redirects=True)
        if r.status_code < 400:
            return f"Live ({r.status_code})"
        else:
            return f"Broken ({r.status_code})"
    except requests.exceptions.SSLError:
        # retry without verifying cert — flags weak SSL but site may still be up
        try:
            r = requests.get(u, headers=HEADERS, timeout=timeout, allow_redirects=True, verify=False)
            return f"Live, bad SSL ({r.status_code})"
        except Exception as e:
            return f"Error: {type(e).__name__}"
    except requests.exceptions.Timeout:
        return "Timeout"
    except requests.exceptions.ConnectionError:
        return "Connection failed"
    except Exception as e:
        return f"Error: {type(e).__name__}"

results = {}
rows_with_url = df[df['Website'].notna() & (df['Website'].astype(str).str.strip() != '')]
print(f"Checking {len(rows_with_url)} websites...")

with ThreadPoolExecutor(max_workers=10) as executor:
    future_to_idx = {
        executor.submit(check_url, row['Website']): idx
        for idx, row in rows_with_url.iterrows()
    }
    for future in as_completed(future_to_idx):
        idx = future_to_idx[future]
        results[idx] = future.result()
        print(f"{df.loc[idx, 'Instructor Name']:25s} {df.loc[idx, 'Website'][:50]:50s} -> {results[idx]}")

# Cell 4 — write results back and save
df['Website Status'] = df.index.map(lambda i: results.get(i, 'No URL'))

live = df['Website Status'].str.startswith('Live').sum()
broken = df['Website Status'].str.startswith('Broken').sum()
no_url = (df['Website Status'] == 'No URL').sum()
other = len(df) - live - broken - no_url
print(f"\nLive: {live} | Broken: {broken} | No URL: {no_url} | Errors/Timeouts: {other}")

out_name = "Pilates_Directory_WithWebsiteCheck.xlsx"
df.to_excel(out_name, index=False)

# Cell 5 — download
files.download(out_name)

Saving Pilates_Directory_Deduped.xlsx to Pilates_Directory_Deduped (1).xlsx
Checking 95 websites...
Amy Curtis                https://www.amycurtis.co.uk/                       -> Live (200)
Alison Cobley             https://www.pilateswithalison.net/                 -> Live (200)
Angela Porter             https://www.facebook.com/risepilatesspringfield    -> Broken (400)
Amanda Jagger             https://www.pilates4.com/                          -> Live (200)
Amanda Cooper             https://www.amandapilates.com/                     -> Live (200)
Agnes Moser               https://www.agnesmpilates.com/                     -> Live (200)
Cat Hyde                  https://olivetreepilates.com/                      -> Live (200)
Catherine Court           https://www.cathycourtpilates.com/                 -> Live (200)
Brenda Nassali-Liston     https://www.informpilates.co.uk/                   -> Live (200)
Charlotte Murray          https://www.charlottemurraypilates.co.uk/          ->

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ── Add "Social?" and "Platform Type" columns based on the Website URL ──
# Run in Colab. Upload your Pilates_Directory_WithWebsiteCheck.xlsx when prompted.

# Cell 1 — install/import
!pip install openpyxl pandas -q

import pandas as pd
from google.colab import files

# Cell 2 — upload the file
uploaded = files.upload()  # pick Pilates_Directory_WithWebsiteCheck.xlsx
xlsx_name = list(uploaded.keys())[0]
df = pd.read_excel(xlsx_name)

# Cell 3 — classify each URL
def classify(url):
    if pd.isna(url) or not str(url).strip():
        return ('No', '')
    u = str(url).lower()
    if 'facebook.com' in u:
        return ('Yes', 'Facebook')
    if 'instagram.com' in u:
        return ('Yes', 'Instagram')
    return ('No', '')

df[['Social?', 'Platform Type']] = df['Website'].apply(lambda u: pd.Series(classify(u)))

print(df['Social?'].value_counts())
print(df[df['Social?'] == 'Yes'][['Instructor Name', 'Website', 'Social?', 'Platform Type']])

# Cell 4 — save and download
out_name = "Pilates_Directory_WithSocialFlag.xlsx"
df.to_excel(out_name, index=False)
files.download(out_name)

Saving Pilates_Directory_WithWebsiteCheck.xlsx to Pilates_Directory_WithWebsiteCheck (1).xlsx
Social?
No     146
Yes      4
Name: count, dtype: int64
        Instructor Name                                            Website  \
9         Angela Porter    https://www.facebook.com/risepilatesspringfield   
31          Daria Moria  https://www.instagram.com/daria.movement.pilates/   
62         Julia Clarke              https://www.facebook.com/helixpilates   
75  Kirsty Wachuku-King         https://www.facebook.com/pilatespassionuk/   

   Social? Platform Type  
9      Yes      Facebook  
31     Yes     Instagram  
62     Yes      Facebook  
75     Yes      Facebook  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ── Extract postcode + classify region (Essex/London) using Gemini ──
# FIXED VERSION: works with the real column name "Locations", which can contain
# MULTIPLE addresses in one cell, separated by " | " (see "Number of Locations").
# For every row, we process EACH location individually and rejoin the results
# with " | " so they stay aligned 1-to-1 with the original Locations list.
#
# Design:
#   1. Postcode is extracted with REGEX first (deterministic) — per location.
#   2. Region is decided by checking for literal "Essex"/"London" text FIRST
#      (deterministic, saves API calls), per location. Gemini is only called for
#      the individual locations where neither word appears literally — and for
#      those, Gemini uses its own knowledge of UK geography/towns/postcode areas
#      to answer Essex, London, or Other (anywhere else in the UK). "Unclear" is
#      reserved only for addresses with genuinely insufficient information.
#   3. At the end, every location with a PARTIAL postcode (outward code only,
#      e.g. "BR2" with no inward part) is printed in a manual-review list.
# Run in Colab. Upload your latest xlsx when prompted.

# Cell 1 — install/import
!pip install -q -U google-generativeai openpyxl pandas
!pip install -q -U google-genai  # newer SDK, used below

import pandas as pd
import re
import time
from google.colab import files
from getpass import getpass

# Cell 2 — upload the file
uploaded = files.upload()  # pick your latest Pilates_Directory_....xlsx
xlsx_name = list(uploaded.keys())[0]
df = pd.read_excel(xlsx_name)

# Cell 3 — enter your Gemini API key (hidden input, not saved to the notebook)
GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

from google import genai
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-3.5-flash"  # gemini-2.0-flash was shut down by Google on June 1, 2026

LOCATION_COL = "Locations"   # <-- the real column name in this file
DELIM = " | "                # <-- how multiple locations are joined in one cell

# Cell 4 — deterministic UK postcode extraction (regex, no LLM involved)
FULL_PC_RE = re.compile(
    r'\b([A-Z]{1,2}\d[A-Z\d]?)\s*(\d[A-Z]{2})\b', re.IGNORECASE
)
OUTWARD_ONLY_RE = re.compile(r'\b([A-Z]{1,2}\d[A-Z\d]?)\b', re.IGNORECASE)

def extract_postcode(address):
    if pd.isna(address) or str(address).strip() == '':
        return ('', 'No')
    addr = str(address)
    full_match = FULL_PC_RE.search(addr)
    if full_match:
        pc = f"{full_match.group(1).upper()} {full_match.group(2).upper()}"
        return (pc, 'Yes')
    outward_match = OUTWARD_ONLY_RE.search(addr)
    if outward_match:
        return (outward_match.group(1).upper(), 'No')
    return ('', 'No')

# Cell 5 — deterministic region check: literal "Essex" / "London" in the address text
def literal_region(address):
    if pd.isna(address) or str(address).strip() == '':
        return None
    addr = str(address).lower()
    has_essex = 'essex' in addr
    has_london = 'london' in addr
    if has_essex and not has_london:
        return 'Essex'
    if has_london and not has_essex:
        return 'London'
    return None  # ambiguous (both) or neither present — let Gemini try

# Cell 6 — Gemini fallback ONLY for individual locations where the literal check found nothing.
# This version lets Gemini use its own knowledge of UK geography, towns, and postcode areas
# (not just the literal words "Essex"/"London" in the text) to decide the region.
# We only care about flagging Essex vs London — everywhere else in the UK is "Other".
# "Unclear" is reserved for the rare case where there truly isn't enough info to place
# the address anywhere (e.g. blank, garbled, or no location info at all).
PROMPT_TEMPLATE = """You are classifying a UK address into a region.

Address: "{address}"

Use your knowledge of UK geography, towns, and postcode areas to decide which region this
address falls in, even if the words "Essex" or "London" do not literally appear in the text.

Rules:
- If the address is in Essex (or a town/postcode area that is part of Essex), answer "Essex".
- If the address is in London (or a town/postcode area that is part of Greater London), answer "London".
- If the address is confidently placed somewhere in the UK that is NOT Essex and NOT London,
  answer "Other".
- Only answer "Unclear" if there truly isn't enough information to place the address anywhere
  (e.g. the text is blank, garbled, or gives no usable location information at all).

Respond with ONLY one word: Essex, London, Other, or Unclear. No explanation, no punctuation."""

def gemini_classify(address, retries=3):
    for attempt in range(retries):
        try:
            resp = client.models.generate_content(
                model=MODEL,
                contents=PROMPT_TEMPLATE.format(address=address),
            )
            answer = resp.text.strip()
            if answer in ('Essex', 'London', 'Other', 'Unclear'):
                return answer
            return 'Unclear'
        except Exception as e:
            if attempt == retries - 1:
                print(f"Failed for '{address}': {e}")
                return 'Unclear'
            time.sleep(2 ** attempt)
    return 'Unclear'

# Cell 7 — process every location in every row, then rejoin pipe-delimited
postcodes_out = []
complete_out = []
region_out = []

gemini_calls = 0
literal_hits = 0

for row_idx, row in df.iterrows():
    raw = row[LOCATION_COL]
    if pd.isna(raw) or str(raw).strip() == '':
        postcodes_out.append('')
        complete_out.append('')
        region_out.append('')
        continue

    locations = [loc.strip() for loc in str(raw).split(DELIM)]

    row_postcodes = []
    row_complete = []
    row_regions = []

    for loc in locations:
        pc, complete = extract_postcode(loc)
        row_postcodes.append(pc)
        row_complete.append(complete)

        region = literal_region(loc)
        if region is not None:
            literal_hits += 1
        else:
            region = gemini_classify(loc)
            gemini_calls += 1
            time.sleep(0.5)  # gentle rate limiting
        row_regions.append(region)

    postcodes_out.append(DELIM.join(row_postcodes))
    complete_out.append(DELIM.join(row_complete))
    region_out.append(DELIM.join(row_regions))

    print(f"Row {row_idx:3d} | {row['Instructor Name']:25s} -> "
          f"{len(locations)} location(s), regions: {DELIM.join(row_regions)}")

df['Postcode'] = postcodes_out
df['Postcode Complete'] = complete_out
df['Region'] = region_out

# Cell 8 — summary
print("\n--- Summary ---")
print(f"Literal region matches: {literal_hits}")
print(f"Gemini calls made:      {gemini_calls}")

# Cell 9 — manual review list: every individual location with a PARTIAL postcode
# (outward code only, e.g. "BR2" with no inward part, "Postcode Complete" == "No")
print("\n--- Locations with a PARTIAL postcode (for manual review) ---")
partial_count = 0
for row_idx, row in df.iterrows():
    raw = row[LOCATION_COL]
    if pd.isna(raw) or str(raw).strip() == '':
        continue
    locations = [loc.strip() for loc in str(raw).split(DELIM)]
    row_postcodes = str(row['Postcode']).split(DELIM)
    row_complete = str(row['Postcode Complete']).split(DELIM)
    for loc, pc, complete in zip(locations, row_postcodes, row_complete):
        if complete.strip() == 'No':
            partial_count += 1
            print(f"[{row['Instructor Name']}] postcode='{pc}' -> {loc}")

print(f"\nTotal locations needing manual postcode review: {partial_count}")

# Cell 10 — save
out_name = "Pilates_Directory_WithPostcodeRegion.xlsx"
df.to_excel(out_name, index=False)
files.download(out_name)

Saving Pilates_Directory_WithSocialFlag.xlsx to Pilates_Directory_WithSocialFlag (5).xlsx
Enter your Gemini API key: ··········
Row   0 | Agnes Moser               -> 1 location(s), regions: London
Row   1 | Alison Cobley             -> 1 location(s), regions: Other
Row   2 | Alison Joyce              -> 1 location(s), regions: Other
Row   3 | Alyth Black               -> 8 location(s), regions: Other | London | Other | Other | Other | Other | Other | Other
Row   4 | Amanda Cooper             -> 9 location(s), regions: Other | Other | Other | Other | Other | Other | Other | Other | Other
Row   5 | Amanda Hall               -> 1 location(s), regions: Other
Row   6 | Amanda Jagger             -> 1 location(s), regions: London
Row   7 | Amy Curtis                -> 1 location(s), regions: London
Row   8 | Andie Shelton             -> 5 location(s), regions: Other | Other | Other | Other | Other
Row   9 | Angela Porter             -> 1 location(s), regions: Essex
Row  10 | Anna Anscombe   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ── Filter directory to keep ONLY Essex/London locations ──
# For each profile, keeps only the individual locations (and their matching
# Postcode / Postcode Complete / Region entries) where Region is "Essex" or
# "London". Drops any location tagged "Other" or "Unclear". If a profile ends
# up with NO Essex/London locations at all, the entire row is deleted.
#
# Input: the xlsx that already has Postcode / Postcode Complete / Region columns
# (i.e. the output of the earlier postcode+region extraction script).
# Run in Colab. Upload that file when prompted.

# Cell 1 — install/import
!pip install -q -U openpyxl pandas

import pandas as pd
from google.colab import files

# Cell 2 — upload the file (the one that already has Postcode/Region columns)
uploaded = files.upload()
xlsx_name = list(uploaded.keys())[0]
df = pd.read_excel(xlsx_name)

DELIM = " | "

# Cell 3 — filter each row down to only Essex/London locations
kept_locations = []
kept_postcodes = []
kept_complete = []
kept_regions = []
rows_to_drop = []

for idx, row in df.iterrows():
    locs = str(row['Locations']).split(DELIM)
    pcs = str(row['Postcode']).split(DELIM)
    comps = str(row['Postcode Complete']).split(DELIM)
    regs = str(row['Region']).split(DELIM)

    keep_idx = [i for i, r in enumerate(regs) if r.strip() in ('Essex', 'London')]

    if not keep_idx:
        # no Essex/London location on this profile at all -> drop the whole row
        rows_to_drop.append(idx)
        kept_locations.append(None)
        kept_postcodes.append(None)
        kept_complete.append(None)
        kept_regions.append(None)
        continue

    kept_locations.append(DELIM.join(locs[i].strip() for i in keep_idx))
    kept_postcodes.append(DELIM.join(pcs[i].strip() for i in keep_idx))
    kept_complete.append(DELIM.join(comps[i].strip() for i in keep_idx))
    kept_regions.append(DELIM.join(regs[i].strip() for i in keep_idx))

df['Locations'] = kept_locations
df['Postcode'] = kept_postcodes
df['Postcode Complete'] = kept_complete
df['Region'] = kept_regions
df['Number of Locations'] = df['Region'].apply(
    lambda r: len(str(r).split(DELIM)) if r else 0
)

# Cell 4 — drop rows with no Essex/London locations at all
rows_before = len(df)
df = df.drop(index=rows_to_drop).reset_index(drop=True)
rows_after = len(df)

print(f"Rows before: {rows_before}")
print(f"Rows dropped (no Essex/London location): {len(rows_to_drop)}")
print(f"Rows after: {rows_after}")

# Cell 5 — save + download
out_name = "Pilates_Directory_EssexLondon_Only.xlsx"
df.to_excel(out_name, index=False)
files.download(out_name)

In [ ]:
# ── Apply manually-verified corrections from the postcode review ──
# This applies every fix found while manually reviewing the partial-postcode
# flags on the Essex/London-only directory:
#   - Daria Moria      -> replaced with her 2 real Greenwich locations
#   - Donna Silburn     -> fixed "SW4 OHN" typo to "SW4 0HN"
#   - Imogen Croft      -> replaced with her 3 real class locations
#   - Nishka Smith      -> replaced with her 3 real teaching locations
#   - Victoria Lamb     -> replaced with her 3 real teaching locations,
#                          plus updated website/email/social info
#
# Input: Pilates_Directory_EssexLondon_Only.xlsx (the Essex/London-filtered file)
# Run in Colab. Upload that file when prompted.

# Cell 1 — install/import
!pip install -q -U openpyxl pandas

import pandas as pd
from google.colab import files

# Cell 2 — upload the file
uploaded = files.upload()
xlsx_name = list(uploaded.keys())[0]
df = pd.read_excel(xlsx_name)

DELIM = " | "

def set_locations(df, instructor_name, locations, postcodes, region="London"):
    """Overwrite Locations/Postcode/Postcode Complete/Region/Number of Locations
    for a given instructor with a fresh, fully-verified list."""
    idx = df[df['Instructor Name'] == instructor_name].index[0]
    df.at[idx, 'Locations'] = DELIM.join(locations)
    df.at[idx, 'Postcode'] = DELIM.join(postcodes)
    df.at[idx, 'Postcode Complete'] = DELIM.join(['Yes'] * len(locations))
    df.at[idx, 'Region'] = DELIM.join([region] * len(locations))
    df.at[idx, 'Number of Locations'] = len(locations)
    return idx

# Cell 3 — Daria Moria: replace with her 2 verified Greenwich locations
set_locations(
    df, 'Daria Moria',
    locations=[
        'Market Studios, 18-23 Greenwich Market, Durnford Street, Greenwich, London, SE10 9HZ, GBR',
        'Studio 225, 225 Greenwich High Rd, Greenwich, London, SE10 8NB, GBR',
    ],
    postcodes=['SE10 9HZ', 'SE10 8NB'],
)

# Cell 4 — Donna Silburn: fix the "SW4 OHN" typo (O -> 0) for the North Street location
idx = df[df['Instructor Name'] == 'Donna Silburn'].index[0]
locs = str(df.at[idx, 'Locations']).split(DELIM)
pcs = str(df.at[idx, 'Postcode']).split(DELIM)
comps = str(df.at[idx, 'Postcode Complete']).split(DELIM)
target_i = [i for i, l in enumerate(locs) if 'North Street' in l][0]
locs[target_i] = locs[target_i].replace('SW4 OHN', 'SW4 0HN')
pcs[target_i] = 'SW4 0HN'
comps[target_i] = 'Yes'
df.at[idx, 'Locations'] = DELIM.join(locs)
df.at[idx, 'Postcode'] = DELIM.join(pcs)
df.at[idx, 'Postcode Complete'] = DELIM.join(comps)

# Cell 5 — Imogen Croft: replace with her 3 real class locations (from her timetable)
set_locations(
    df, 'Imogen Croft',
    locations=[
        "St Alban's Church, Furzedown, Streatham, London, SW16 SRR, GBR",  # kept literally as given
        'St Clement with St Peters Church, East Dulwich, London, SE22 0AY, GBR',
        'St Leonards Church Hall, 8 Tooting Bec Gardens, London, SW16 1RB, GBR',
    ],
    postcodes=['SW16 SRR', 'SE22 0AY', 'SW16 1RB'],
)

# Cell 6 — Nishka Smith: replace with her 3 real teaching locations
set_locations(
    df, 'Nishka Smith',
    locations=[
        'Chantry Studios, Chantry Lane, Bromley, Greater London, BR2 9QL, GBR',
        'Norman Park, Hayes Lane, Bromley, BR2 9EG, GBR',
        'Tula Balance, V22 Studio, The Priory, Church Hill, Orpington, BR6 0HH, GBR',
    ],
    postcodes=['BR2 9QL', 'BR2 9EG', 'BR6 0HH'],
)

# Cell 7 — Victoria Lamb: replace with her 3 real teaching locations
set_locations(
    df, 'Victoria Lamb',
    locations=[
        'Yogahome, 14 Allen Road, London, N16 8SD, GBR',
        'Higham Hill Hub, London, E17 5QT, GBR',
        'Studio M, 2 Hitchcock Business Centre, Station Approach, Leytonstone High Road, London, E11 4RE, GBR',
    ],
    postcodes=['N16 8SD', 'E17 5QT', 'E11 4RE'],
)

# Cell 8 — Victoria Lamb: also update her contact/social info found during review
idx = df[df['Instructor Name'] == 'Victoria Lamb'].index[0]
df.at[idx, 'Website'] = 'https://www.pilateswithvictoria.co.uk/'
df.at[idx, 'Email'] = 'v.berry@hotmail.co.uk'
df.at[idx, 'Platform Type'] = 'Instagram'
df.at[idx, 'Social?'] = 'Yes'

# Cell 9 — save + download
out_name = "Pilates_Directory_EssexLondon_Only.xlsx"
df.to_excel(out_name, index=False)
files.download(out_name)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 71.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.


Saving Pilates_Directory_EssexLondon_Only.xlsx to Pilates_Directory_EssexLondon_Only.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
"""
Geocode the Pilates directory postcodes using postcodes.io.

Run this in Colab (or any environment with internet access — it will NOT
work in a sandboxed environment without outbound access to api.postcodes.io).

Steps:
1. Upload Pilates_Directory_EssexLondon_Only__1_.xlsx to your Colab session
   (or mount Drive and point INPUT_PATH at it).
2. pip install openpyxl requests  (usually already available in Colab)
3. Run this script — it will produce Pilates_Directory_EssexLondon_geocoded.xlsx
"""

import openpyxl
import requests
import time

INPUT_PATH = "/content/Pilates_Directory_EssexLondon_Only (1).xlsx"
OUTPUT_PATH = "Pilates_Directory_EssexLondon_geocoded.xlsx"
POSTCODE_COL_NAME = "Postcode"  # column that may contain "PC1 | PC2 | PC3"
BULK_ENDPOINT = "https://api.postcodes.io/postcodes"
BATCH_SIZE = 100  # postcodes.io bulk limit

wb = openpyxl.load_workbook(INPUT_PATH)
ws = wb.active

headers = [cell.value for cell in ws[1]]
postcode_col_idx = headers.index(POSTCODE_COL_NAME)  # 0-based

# --- 1. Collect every individual postcode across all rows (split on "|") ---
all_postcodes = set()
row_postcode_lists = []  # per-row list of cleaned postcodes, in original order

for row in ws.iter_rows(min_row=2, values_only=True):
    raw = row[postcode_col_idx]
    if raw:
        pcs = [p.strip() for p in str(raw).split("|") if p.strip()]
    else:
        pcs = []
    row_postcode_lists.append(pcs)
    all_postcodes.update(pcs)

all_postcodes = list(all_postcodes)
print(f"Found {len(all_postcodes)} unique postcodes across {len(row_postcode_lists)} rows.")

# --- 2. Bulk geocode in batches of 100 ---
lookup = {}  # postcode -> (lat, lon) or (None, None) if not found

for i in range(0, len(all_postcodes), BATCH_SIZE):
    batch = all_postcodes[i:i + BATCH_SIZE]
    resp = requests.post(BULK_ENDPOINT, json={"postcodes": batch}, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    for item in data["result"]:
        pc = item["query"]
        result = item["result"]
        if result:
            lookup[pc] = (result["latitude"], result["longitude"])
        else:
            lookup[pc] = (None, None)
            print(f"  Not found: {pc}")

    time.sleep(0.2)  # be polite to the API

print(f"Geocoded {sum(1 for v in lookup.values() if v[0] is not None)} / {len(all_postcodes)} successfully.")

# --- 3. Write Latitude / Longitude columns back, pipe-separated to match order ---
lat_col = len(headers) + 1
lon_col = len(headers) + 2
ws.cell(row=1, column=lat_col, value="Latitude")
ws.cell(row=1, column=lon_col, value="Longitude")

for row_idx, pcs in enumerate(row_postcode_lists, start=2):
    lats, lons = [], []
    for pc in pcs:
        lat, lon = lookup.get(pc, (None, None))
        lats.append(str(lat) if lat is not None else "")
        lons.append(str(lon) if lon is not None else "")
    ws.cell(row=row_idx, column=lat_col, value=" | ".join(lats))
    ws.cell(row=row_idx, column=lon_col, value=" | ".join(lons))

wb.save(OUTPUT_PATH)
print(f"Saved: {OUTPUT_PATH}")

Found 131 unique postcodes across 75 rows.
  Not found: SW16 SRR
Geocoded 130 / 131 successfully.
Saved: Pilates_Directory_EssexLondon_geocoded.xlsx


In [3]:
"""
Fix the "SW16 SRR" typo (Imogen Croft row) -> "SW16 6RR" and geocode it,
filling in the missing Latitude/Longitude in the already-geocoded file.

Run this in Colab, in the same folder as
Pilates_Directory_EssexLondon_geocoded.xlsx (the output of the previous script).

pip install openpyxl requests   # if not already available
"""

import openpyxl
import requests

INPUT_PATH = "Pilates_Directory_EssexLondon_geocoded.xlsx"
OUTPUT_PATH = "Pilates_Directory_EssexLondon_geocoded_fixed.xlsx"

OLD_POSTCODE = "SW16 SRR"
NEW_POSTCODE = "SW16 6RR"

wb = openpyxl.load_workbook(INPUT_PATH)
ws = wb.active

headers = [cell.value for cell in ws[1]]
postcode_col = headers.index("Postcode") + 1  # 1-based for openpyxl
lat_col = headers.index("Latitude") + 1
lon_col = headers.index("Longitude") + 1

# --- 1. Find the row with the typo and fix the postcode ---
target_row = None
part_index = None

for row_idx in range(2, ws.max_row + 1):
    raw = ws.cell(row=row_idx, column=postcode_col).value
    if raw and OLD_POSTCODE in raw:
        parts = [p.strip() for p in raw.split("|")]
        part_index = parts.index(OLD_POSTCODE)
        parts[part_index] = NEW_POSTCODE
        ws.cell(row=row_idx, column=postcode_col, value=" | ".join(parts))
        target_row = row_idx
        break

if target_row is None:
    raise ValueError(f"Could not find '{OLD_POSTCODE}' in the Postcode column.")

print(f"Fixed postcode in row {target_row}: '{OLD_POSTCODE}' -> '{NEW_POSTCODE}' "
      f"(position {part_index} in that row's list)")

# --- 2. Geocode the corrected postcode ---
resp = requests.get(f"https://api.postcodes.io/postcodes/{NEW_POSTCODE.replace(' ', '')}", timeout=15)
resp.raise_for_status()
data = resp.json()

if data["result"] is None:
    raise ValueError(f"postcodes.io could not resolve '{NEW_POSTCODE}' either — check the postcode.")

lat = data["result"]["latitude"]
lon = data["result"]["longitude"]
print(f"Geocoded {NEW_POSTCODE}: lat={lat}, lon={lon}")

# --- 3. Slot the lat/lon into the right position in that row's pipe-separated list ---
lat_raw = ws.cell(row=target_row, column=lat_col).value or ""
lon_raw = ws.cell(row=target_row, column=lon_col).value or ""

lats = [p.strip() for p in lat_raw.split("|")] if lat_raw else []
lons = [p.strip() for p in lon_raw.split("|")] if lon_raw else []

# Pad in case the row previously had fewer entries than positions (shouldn't happen, but safe)
while len(lats) <= part_index:
    lats.append("")
while len(lons) <= part_index:
    lons.append("")

lats[part_index] = str(lat)
lons[part_index] = str(lon)

ws.cell(row=target_row, column=lat_col, value=" | ".join(lats))
ws.cell(row=target_row, column=lon_col, value=" | ".join(lons))

wb.save(OUTPUT_PATH)
print(f"Saved: {OUTPUT_PATH}")

Fixed postcode in row 27: 'SW16 SRR' -> 'SW16 6RR' (position 0 in that row's list)
Geocoded SW16 6RR: lat=51.424037, lon=-0.144111
Saved: Pilates_Directory_EssexLondon_geocoded_fixed.xlsx


In [4]:
"""
Replace the placeholder "Verified" in the Layer 2 - Pre/Postnatal Specialism
column with the actual qualification title + awarding body, confirmed from
Body Control Pilates' own site:

    https://www.bodycontrolpilates.com/find-your-local-teacher/your-guide-to-our-teacher-qualifications/

"Their Pre- & Postnatal Pilates specialism (the Q003 filter used when scraping)
is awarded by Active IQ, ratified by Ofqual, full title:
'Level 3 Award in Designing Pre- and Postnatal Exercise Programmes'."

Run this in Colab, in the same folder as
Pilates_Directory_EssexLondon_geocoded_fixed.xlsx (or whatever your latest
version is named — update INPUT_PATH below).

pip install openpyxl   # if not already available
"""

import openpyxl

INPUT_PATH = "Pilates_Directory_EssexLondon_geocoded_fixed.xlsx"
OUTPUT_PATH = "Pilates_Directory_EssexLondon_layer2_labelled.xlsx"

LAYER2_COL_NAME = "Layer 2 - Pre/Postnatal Specialism"
OLD_VALUE = "Verified"
NEW_VALUE = "Level 3 Award in Designing Pre- and Postnatal Exercise Programmes (Active IQ, Ofqual-regulated)"

wb = openpyxl.load_workbook(INPUT_PATH)
ws = wb.active

headers = [cell.value for cell in ws[1]]
layer2_col = headers.index(LAYER2_COL_NAME) + 1  # 1-based for openpyxl

updated = 0
skipped = 0

for row_idx in range(2, ws.max_row + 1):
    cell = ws.cell(row=row_idx, column=layer2_col)
    if cell.value == OLD_VALUE:
        cell.value = NEW_VALUE
        updated += 1
    elif cell.value:
        skipped += 1
        print(f"  Row {row_idx}: unexpected existing value, left as-is -> {cell.value!r}")

print(f"Updated {updated} rows.")
if skipped:
    print(f"Skipped {skipped} rows with a different existing value (see above).")

wb.save(OUTPUT_PATH)
print(f"Saved: {OUTPUT_PATH}")

Updated 75 rows.
Saved: Pilates_Directory_EssexLondon_layer2_labelled.xlsx
